# EntityLens AI — Transformer Fine-tuning (Kaggle GPU)

Fine-tunes **bert-base-cased** for token-level NER on the frozen CoNLL-2003
splits shipped with the code dataset. Produces strict entity-level seqeval
metrics and saves the model + tokenizer + label maps for deployment.

**Runtime:** set Accelerator = GPU (T4/P100). The frozen parquet splits travel
in the attached `entitylens-ai-code` dataset, so no re-download and no secrets.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,"-m","pip","install","-q",
    "seqeval==1.2.2","transformers==4.46.3","accelerate==1.1.1","datasets==2.21.0"],check=True)

In [ ]:
from pathlib import Path
# The code dataset is mounted read-only under /kaggle/input/<slug>.
CAND = list(Path('/kaggle/input').glob('*/src/entitylens'))
BASE = CAND[0].parents[1] if CAND else Path('.')
sys.path.insert(0, str(BASE/'src'))
PROCESSED = BASE/'data'/'processed'
print('BASE', BASE, '| processed exists:', PROCESSED.exists())

In [ ]:
import torch, json
from entitylens.config import load_config
from entitylens.training.transformer_trainer import fine_tune
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
cfg = load_config(BASE/'configs'/'transformer.yaml', BASE/'configs'/'base.yaml')
print('config:', cfg)

In [ ]:
OUT = Path('/kaggle/working/artifacts/transformer')
summary = fine_tune(cfg, PROCESSED, OUT, smoke=False)
print(json.dumps(summary, indent=2))

In [ ]:
# Evaluate on the held-out TEST split (strict seqeval) + latency/size.
from entitylens.inference.predictor import Predictor
from entitylens.evaluation.metrics import entity_level_report, bio_transition_validity
from entitylens.data.dataset import load_split
from entitylens.labels import ID2LABEL
import time
pred = Predictor.load(OUT/'model')
test = load_split(PROCESSED,'test')
yt=[[ID2LABEL[i] for i in e['ner_tags']] for e in test]
yp=[pred.predict_pretokenized(e['tokens']) for e in test]
rep = entity_level_report(yt, yp)
rep['bio_validity']=bio_transition_validity(yp)
(OUT/'test_report.json').write_text(json.dumps(rep, indent=2))
print('TEST F1', rep['micro']['f1'], '| per-type', {k:round(v['f1'],4) for k,v in rep['per_type'].items()})

The saved model lives in `/kaggle/working/artifacts/transformer/model`
(config.json carries `id2label`/`label2id`). Download the output, place it under
`artifacts/transformer/` locally, then run `scripts/evaluate_all.py` to fold it
into the four-model comparison.